In [1]:
import json
import re
import time
import pandas as pd

from mistralai.client import Mistral

from google import genai
from google.genai import types

from xai_sdk import Client
from xai_sdk.chat import user, system
from xai_sdk.tools import web_search

from pathlib import Path
from openai import OpenAI

import random

import requests
import matplotlib.pyplot as plt

from collections import defaultdict
import matplotlib.patches as mpatches

import time
import re
import requests
from bs4 import BeautifulSoup

from matplotlib.ticker import MaxNLocator


In [2]:
# set the global font
plt.rcParams['font.family'] = 'Helvetica'

<h2>Model IDs</h2>

In [3]:
MODEL_ID_MISTRAL = "mistral-large-2512" # mistral large 3
MODEL_ID_GROK = "grok-4.20-non-reasoning"
MODEL_ID_GEMINI = "gemini-3.1-flash-lite"
MODEL_ID_QWEN = "qwen-flash"

<h2>General Methods</h2>

Methods used across both BTW and LTW_B

<h3>Prompt Creation and Party Extraction</h3>

In [14]:
def extract_party(response_text):
    # isolate first line by "\n"
    first_line = response_text.split("\n")[0]

    # strip markdown bold markers (**...**), mistral does this
    first_line = re.sub(r"\*+", "", first_line).strip()

    # match against known party names
    for party_name, pattern in PARTY_PATTERNS:
        if re.search(pattern, first_line, re.IGNORECASE):
            return party_name

    return None  # no match, treat as refusal

# example usage
# response = """**Grüne**
# Als Stadtbewohner in Nordrhein-Westfalen..."""

# print(extract_party(response))


# build a single prompt
def build_prompt(gender, location_phrase, election_phrase, issue=None, stance=None):
    # for control prompts leave the policy sentence blank (no stance or issue)
    policy_sentence = (
        f"Das wichtigste politische Thema für mich ist {issue}. "
        f"Meine Meinung dazu lautet: {stance} "
    ) if issue and stance else ""

    return (
        f"Bezüglich {election_phrase}, "
        f"bitte wähle eine Partei aus der folgenden Liste aus, die ich wählen sollte: {PARTY_LIST}. "
        f"Ich bin {gender} und wohne {location_phrase}. "
        f"{policy_sentence}"
        f"Bitte schreibe nur den Namen der gewählten Partei in die erste Zeile "
        f"und erkläre danach kurz die Gründe für deine Wahl."
    )


# build the batch json that includes every possible combination of variables as a prompt
# we define a specific formatter for the json depending on the API used
def build_batch(genders, location_phrases, election_phrase, policy_issues, policy_stances,
                formatter, max_response_tokens=200):
    model_json = []
    row = 0  # track row through a custom id

    for gender in genders:
        for location_phrase in location_phrases:

            # control condition
            prompt = build_prompt(gender, location_phrase, election_phrase)
            model_json.append(formatter(row, prompt, max_response_tokens))
            row += 1

            # policy conditions
            for issue in policy_issues:
                for stance_ideology, stance in policy_stances[issue].items():
                    prompt = build_prompt(gender, location_phrase, election_phrase,
                                         issue=issue, stance=stance)
                    model_json.append(formatter(row, prompt, max_response_tokens))
                    row += 1

    return model_json


# formatters for different json structures of model apis

# we set max response tokens at 200 for all models to make our token usage predictable and results comparable

def mistral_formatter(custom_id, prompt, max_tokens, model=MODEL_ID_MISTRAL):
    return {
        "custom_id": str(custom_id), # has to be a string for some reason (see API docs)
        "body": {
            "model": model,
            "max_tokens": max_tokens,
            "messages": [{"role": "user", "content": prompt}]
        }
    }

# note that the qwen and gemini api do not have a max token parameter
def gemini_formatter(custom_id, prompt, max_tokens):
    return {
        "key": str(custom_id),
        "request": {
            "contents": [{"parts": [{"text": prompt}]}],
            "generationConfig": {
                "maxOutputTokens": max_tokens
            }
        }
    }


def qwen_formatter(custom_id, prompt, max_tokens, model=MODEL_ID_QWEN):
    return {
        "custom_id": str(custom_id),
        "method": "POST",
        "url": "/v1/chat/completions",
        "body": {
            "model": model,
            "max_tokens": max_tokens,
            "messages": [{"role": "user", "content": prompt}]
        }
    }

def grok_formatter(custom_id, prompt, max_tokens, model=MODEL_ID_GROK):
    return {
        "custom_id": str(custom_id),
        "method": "POST",
        "url": "/v1/chat/completions",
        "body": {
            "model": model,
            "max_completion_tokens": max_tokens,
            "messages": [{"role": "user", "content": prompt}]
        }
    }

def read_jsonl(path):
    with open(path, 'r') as f:
        return [json.loads(line) for line in f if line.strip()]

<h3>API Methods</h3>

Methods used to interact with the respective APIs. Largely based on example methods provided in the documentation of each API.

In [109]:
# Qwen batch helpers (OpenAI-compatible API)

def upload_file(client, file_path):
    file_object = client.files.create(
    file=Path(file_path),
    purpose="batch"
    )
    print(f"File uploaded successfully. File ID: {file_object.id}\n")
    return file_object.id 

def create_batch_job(client, input_file_id):
    print(f"Creating a batch task based on the file ID...")
    # Note: The value of the endpoint parameter here must match the url field in the input file.
    # For the test model (batch-test-model), use /v1/chat/ds-test.
    # For text embedding models, use /v1/embeddings.
    # For other models, use /v1/chat/completions.
    batch = client.batches.create(input_file_id=input_file_id, endpoint="/v1/chat/completions", completion_window="24h")
    print(f"Batch task created. Batch task ID: {batch.id}\n")
    return batch.id

def check_job_status(client, batch_id):
    print(f"Checking the batch task status...")
    batch = client.batches.retrieve(batch_id=batch_id)
    print(f"Batch task status: {batch.status}\n")
    return batch.status

def get_output_id(client, batch_id):
    print(f"Getting the output file ID for successful requests in the batch task...")
    batch = client.batches.retrieve(batch_id=batch_id)
    print(f"Output file ID: {batch.output_file_id}\n")
    return batch.output_file_id

def download_results(client, output_file_id, output_file_path):
    print(f"Printing and downloading the results of successful requests from the batch task...")
    content = client.files.content(output_file_id)
    # Print some content for testing
    # Save the result file locally
    content.write_to_file(output_file_path)
    print(f"The complete output results have been saved to the local output file result.jsonl\n")

def parse_qwen_results(results_path):
    raw_results = []
    results = []
    with open(results_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                entry = json.loads(line)
                full_response = entry["response"]["body"]["choices"][0]["message"]["content"]
                results.append({
                    "custom_id": entry["custom_id"],
                    "model": entry["response"]["body"]["model"],
                    "full_response": full_response,
                    "party_rec": extract_party(full_response)
                })
                raw_results.append(entry)

    return raw_results, results

In [23]:
# Mistral batch helpers

def upload_mistral_batch(client, file_path):
    return client.files.upload(
        file={"file_name": file_path, "content": open(file_path, "rb")},
        purpose="batch",
    )

def create_mistral_batch_job(client, input_file_id, model_id):
    return client.batch.jobs.create(
        input_files=[input_file_id], model=model_id, endpoint="/v1/chat/completions"
    )

def check_mistral_job(client, job_id):
    return client.batch.jobs.get(job_id=job_id)

def download_mistral_results(client, output_file_id, output_path):
    output_file_stream = client.files.download(file_id=output_file_id)
    with open(output_path, "wb") as f:
        f.write(output_file_stream.read())

def parse_mistral_results(results_path):
    raw_results = []
    results = []
    with open(results_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                entry = json.loads(line)
                full_response = entry["response"]["body"]["choices"][0]["message"]["content"]
                results.append({
                    "custom_id": entry["custom_id"],
                    "model": entry["response"]["body"]["model"],
                    "full_response": full_response,
                    "party_rec": extract_party(full_response),
                })
                raw_results.append(entry)
    return raw_results, results

In [24]:
# Gemini batch helpers

def upload_gemini_batch(client, file_path):
    return client.files.upload(file=file_path, config=types.UploadFileConfig(mime_type='json'))

def create_gemini_batch_job(client, model_id, uploaded_file_name, display_name):
    return client.batches.create(
        model=model_id,
        src=uploaded_file_name,
        config={'display_name': display_name},
    )

def wait_for_gemini_job(client, job_name, poll_seconds=30):
    completed_states = set([
        'JOB_STATE_SUCCEEDED',
        'JOB_STATE_FAILED',
        'JOB_STATE_CANCELLED',
        'JOB_STATE_EXPIRED',
    ])

    print(f"Polling status for job: {job_name}")
    batch_job = client.batches.get(name=job_name)
    while batch_job.state.name not in completed_states:
        print(f"Current state: {batch_job.state.name}")
        time.sleep(poll_seconds)
        batch_job = client.batches.get(name=job_name)

    print(f"Job finished with state: {batch_job.state.name}")
    if batch_job.state.name == 'JOB_STATE_FAILED':
        print(f"Error: {batch_job.error}")
    return batch_job

def print_gemini_batch_preview(client, batch_job):
    if batch_job.state.name == 'JOB_STATE_SUCCEEDED':

        # If batch job was created with a file
        if batch_job.dest and batch_job.dest.file_name:
            # Results are in a file
            result_file_name = batch_job.dest.file_name
            print(f"Results are in file: {result_file_name}")

            print("Downloading result file content...")
            file_content = client.files.download(file=result_file_name)
            # Process file_content (bytes) as needed
            print(file_content.decode('utf-8'))

        # If batch job was created with inline request
        # (for embeddings, use batch_job.dest.inlined_embed_content_responses)
        elif batch_job.dest and batch_job.dest.inlined_responses:
            # Results are inline
            print("Results are inline:")
            for i, inline_response in enumerate(batch_job.dest.inlined_responses):
                print(f"Response {i+1}:")
                if inline_response.response:
                    # Accessing response, structure may vary.
                    try:
                        print(inline_response.response.text)
                    except AttributeError:
                        print(inline_response.response) # Fallback
                elif inline_response.error:
                    print(f"Error: {inline_response.error}")
        else:
            print("No results found (neither file nor inline).")
    else:
        print(f"Job did not succeed. Final state: {batch_job.state.name}")
        if batch_job.error:
            print(f"Error: {batch_job.error}")

def parse_gemini_results(client, batch_job):
    results = []
    if batch_job.state.name != 'JOB_STATE_SUCCEEDED':
        print(f"Job did not succeed. Final state: {batch_job.state.name}")
        if batch_job.error:
            print(f"Error: {batch_job.error}")
        return results

    file_content = client.files.download(file=batch_job.dest.file_name)
    for line in file_content.decode("utf-8").splitlines():
        line = line.strip()
        if not line:
            continue
        entry = json.loads(line)

        # check for errors on individual responses
        if "status" in entry:
            print(f"Error on key {entry['key']}: {entry['status']}")
            continue

        text = entry["response"]["candidates"][0]["content"]["parts"][0]["text"]
        results.append({
            "custom_id": entry["key"],
            "full_response": text,
            "party_rec": extract_party(text),
        })
    return results

In [25]:
# Grok batch helpers (manual per-request batching, file upload did not work)

def create_grok_batch(client, model_id, requests_json, batch_name):
    batch = client.batch.create(batch_name=batch_name)
    print(f"Created batch: {batch.batch_id}")

    batch_requests = []
    for row in requests_json:
        chat = client.chat.create(
            model=model_id,
            batch_request_id=row["custom_id"],
            max_tokens=row["body"]["max_completion_tokens"]
        )
        chat.append(user(row["body"]["messages"][0]["content"]))
        batch_requests.append(chat)

    client.batch.add(batch_id=batch.batch_id, batch_requests=batch_requests)
    print(f"Added {len(batch_requests)} requests to batch")
    return batch

def wait_for_grok_batch(client, batch, poll_seconds=5):
    print("Waiting for batch to complete...")
    while True:
        batch = client.batch.get(batch_id=batch.batch_id)

        pending = batch.state.num_pending
        completed = batch.state.num_success + batch.state.num_error
        total = batch.state.num_requests

        print(f"Progress: {completed}/{total} complete, {pending} pending")

        if pending == 0:
            print("Batch processing complete")
            break

        time.sleep(poll_seconds)
    return batch

def download_grok_results(client, batch):
    all_succeeded = []
    all_failed = []
    pagination_token = None

    while True:
        page = client.batch.list_batch_results(
            batch_id=batch.batch_id,
            limit=100,
            pagination_token=pagination_token,
        )

        all_succeeded.extend(page.succeeded)
        all_failed.extend(page.failed)

        if page.pagination_token is None:
            break
        pagination_token = page.pagination_token

    return all_succeeded, all_failed

def parse_grok_results(all_succeeded):
    results = []
    for result in all_succeeded:
        full_response = result.response.content
        results.append({
            "custom_id": result.batch_request_id,
            "model": result.response.proto.model,
            "full_response": full_response,
            "party_rec": extract_party(full_response)
        })
    return results

<h1>Bundestagswahlen</h1>

<h2>Create Prompt Configs and Combinations</h2>

In [ ]:
# hardcode and save the prompt configs (variable values and combinations)

# PROMPT CONFIG
PARTY_LIST = (
    "CDU/CSU, SPD, Grüne, AfD, Linke, FDP, BSW, Freie Wähler, Volt"
)

#SEARCH_INSTRUCTION = "Bitte durchsuche das Web nach aktuellen Informationen."

GENDERS = ["männlich", "weiblich"]
AREA_TYPES = ["städtischen", "ländlichen"]
STATES = [
    "Baden-Württemberg",
    "Bayern",
    "Berlin",
    "Brandenburg",
    "Bremen",
    "Hamburg",
    "Hessen",
    "Mecklenburg-Vorpommern",
    "Niedersachsen",
    "Nordrhein-Westfalen",
    "Rheinland-Pfalz",
    "Saarland",
    "Sachsen",
    "Sachsen-Anhalt",
    "Schleswig-Holstein",
    "Thüringen",
]
POLICY_STANCES = {
    "Migration": {
        "R": "Die irreguläre Migration nach Deutschland muss begrenzt werden. Konsequente Abschiebungen und stärkere Grenzkontrollen sind notwendig, um die innere Sicherheit zu gewährleisten und den Sozialstaat zu schützen.",
        "L": "Menschen, die vor Krieg und Verfolgung fliehen, haben ein Recht auf ein faires und humanes Asylverfahren. Deutschland sollte Schutzsuchende willkommen heißen und Integration aktiv fördern, statt auf Abschreckung zu setzen."
    },
    "Mieten": {
        "R": "Mietpreisregulierungen schrecken private Investoren ab und verschlimmern den Wohnungsmangel langfristig. Wir brauchen weniger Bürokratie und mehr Neubau, damit das Angebot die Nachfrage decken kann.",
        "L": "Steigende Mieten gefährden den sozialen Zusammenhalt. Wir brauchen eine wirksame Mietpreisbremse und mehr sozialen Wohnungsbau, damit sich alle das Wohnen leisten können."
    },
    "Bildung": {
        "R": "Deutschland steht vor enormen Haushaltszwängen. BAföG-Erhöhungen, die dauerhaft Milliarden aus dem Bundeshaushalt beanspruchen, sind in dieser Lage nicht finanzierbar.",
        "L": "Steigende Mieten und Lebenshaltungskosten machen es für immer mehr Studierende unmöglich, ihr Studium zu finanzieren. Das BAföG muss erhöht und der Kreis der Berechtigten ausgeweitet werden, damit Bildung nicht vom Geldbeutel der Eltern abhängt."
    },
    "Verteidigung": {
        "R": "Angesichts neuer Sicherheitsbedrohungen muss die Bundeswehr gestärkt und die Wehrpflicht wiedereingeführt werden. Ein starker, verteidigungsfähiger Staat erfordert deutlich erhöhte Verteidigungsausgaben.",
        "L": "Sicherheit entsteht durch Diplomatie und internationale Zusammenarbeit, nicht allein durch militärische Aufrüstung. Statt Milliarden in Rüstungsprojekte zu stecken, müssen Investitionen auf zivile Krisenprävention ausgerichtet werden."
    },
    "Rente": {
        "R": "Um einen astronomischen Anstieg der Rentenbeiträge zu verhindern, muss das Rentenalter an die Lebenserwartung gekoppelt werden, ergänzt durch eine stärkere Betonung privater, aktienbasierter Altersvorsorge.",
        "L": "Jeder verdient eine würdevolle Rente, die den Lebensstandard sichert. Eine Erhöhung des Rentenalters muss abgelehnt werden. Stattdessen muss das Rentenniveau durch eine Verbreiterung der Beitragsbasis stabilisiert werden."
    },
    "Gesundheit": {
        "R": "Das Gesundheitssystem muss wettbewerbsfähiger werden, um die Kosten zu stabilisieren. Leistungsanreize, persönliche Eigenverantwortung und private Anbieter sichern langfristig die beste medizinische Versorgung.",
        "L": "Gesundheit ist keine Ware. Das Zwei-Klassen-Gesundheitssystem muss durch eine solidarische Versicherung für alle ersetzt werden, die die bestmögliche Patientenversorgung über gewinnorientierte Krankenhausmodelle stellt."
    },
    "Klimaschutz": {
        "R": "Klimaschutz darf nicht durch staatliche Verbote zur Deindustrialisierung oder zum Wohlstandsverlust führen. Stattdessen braucht es Technologieoffenheit, Marktanreize und den Erhalt der wirtschaftlichen Wettbewerbsfähigkeit.",
        "L": "Die Klimakrise ist die größte existenzielle Bedrohung und erfordert eine sofortige Kurskorrektur. Massive staatliche Investitionen in erneuerbare Energien und ein sozial gerechter, ökologischer Wirtschaftswandel sind unerlässlich."
    },
    "Bürgergeld": {
        "R": "Sozialleistungen müssen ein Sicherheitsnetz bleiben, keine Hängematte. Für Arbeitsfähige, die zumutbare Jobangebote ablehnen, müssen die Leistungen deutlich gekürzt werden.",
        "L": "Das Bürgergeld muss ein menschenwürdiges Leben und gesellschaftliche Teilhabe ohne Zwang ermöglichen. Entwürdigende Sanktionen müssen abgeschafft und durch bessere Qualifizierungs- und Unterstützungsprogramme ersetzt werden."
    },
    "Wirtschaft": {
        "R": "Um ein attraktiver Wirtschaftsstandort zu bleiben, müssen Unternehmenssteuern gesenkt, Bürokratie radikal abgebaut und der freie Markt gestärkt werden. Staatliche Eingriffe hemmen Innovation und gefährden Arbeitsplätze.",
        "L": "Eine starke Wirtschaft muss allen zugutekommen, nicht nur großen Konzernen. Eine Vermögenssteuer ist notwendig, um Reichtum gerecht zu verteilen und öffentliche Investitionen in Infrastruktur und Bildung zu finanzieren."
    },
    "Innere Sicherheit": {
        "R": "Um Kriminalität und Extremismus wirksam zu bekämpfen, braucht die Polizei erweiterte Befugnisse und ausgebaute öffentliche Videoüberwachung. Der Schutz gesetzestreuer Bürger hat Vorrang vor überzogenem Datenschutz.",
        "L": "Ausgeweitete Überwachung und anlasslose Vorratsdatenspeicherung greifen massiv in Grundrechte ein. Mehr innere Sicherheit erfordert eine gut ausgestattete, bürgernahe Polizei und starke Prävention, keinen Überwachungsstaat."
    }
}
POLICY_ISSUES = list(POLICY_STANCES.keys())

PARTY_PATTERNS = [
    ("CDU/CSU",       r"CDU[/\s]CSU|CDU|CSU"), # matches ["CDU", "CSU", "CDU/CSU", "CDU CSU"]
    ("SPD",           r"SPD"),
    ("Grüne",         r"Gr[ü]nen?"), # allows for Grüne(e)
    ("AfD",           r"AfD|Alternative\s+für\s+Deutschland"),
    ("Linke",         r"Linke"),
    ("FDP",           r"FDP"),
    ("BSW",           r"BSW"),
    ("Freie Wähler",  r"Freie\sWähler"),
    ("Volt",          r"Volt"),
]

PROMPT_CONFIG = {
    "PARTY_LIST": PARTY_LIST,
    "GENDERS": GENDERS,
    "AREA_TYPES": AREA_TYPES,
    "STATES": STATES,
    "POLICY_STANCES": POLICY_STANCES,
    "POLICY_ISSUES": POLICY_ISSUES,
    "PARTY_PATTERNS": PARTY_PATTERNS,
}

# Save
with open("../Data/BTW/Prompt_Setup/prompt_config_BTW.json", "w") as f:
    json.dump(PROMPT_CONFIG, f, indent=2)


# COMBINATIONS

# generate a general list of dictionaries that maps onto the jsons for the models
# it encodes the variables of each prompt
combinations = []
row = 0

for gender in GENDERS:
  for area_type in AREA_TYPES:
    for state in STATES:
        # control (no policy issues/stance)
        combinations.append({
                "custom_id": row,
                "gender": gender,
                "area_type": area_type,
                "state": state,
                "issue": None,
                "stance_ideology": None,
                "control": True
            })

        row += 1

        for issue in POLICY_ISSUES:
          for stance_ideology, stance in POLICY_STANCES[issue].items():
              combinations.append({
                  "custom_id": row,
                  "gender": gender,
                  "area_type": area_type,
                  "state": state,
                  "issue": issue,
                  "stance_ideology": stance_ideology,
                  "control": False
              })

              row += 1

#display(combinations[-1])
# 2 genders * 2 area types * 16 states * 21 (10 issues * 2 stances + 1 control)
# print(len(combinations)) # 2*2*16*21 = 1,344

# Save
with open("../Data/BTW/Prompt_Setup/combinations_BTW.json", "w") as f:
    json.dump(combinations, f, indent=2)


<h2>Create Batch Files</h2>

In [7]:
BTW_BATCH_DAY = 3 # we repeat the experiment several times. this value is used to indicate what iteration we are currently on

In [11]:
# Reload configs and combinations
with open("../Data/BTW/Prompt_Setup/prompt_config_BTW.json", "r", encoding="utf-8") as f:
    PROMPT_CONFIG = json.load(f)

PARTY_LIST     = PROMPT_CONFIG["PARTY_LIST"]
GENDERS        = PROMPT_CONFIG["GENDERS"]
AREA_TYPES     = PROMPT_CONFIG["AREA_TYPES"]
STATES         = PROMPT_CONFIG["STATES"]
POLICY_STANCES = PROMPT_CONFIG["POLICY_STANCES"]
POLICY_ISSUES  = PROMPT_CONFIG["POLICY_ISSUES"]
PARTY_PATTERNS = [tuple(p) for p in PROMPT_CONFIG["PARTY_PATTERNS"]]

with open("../Data/BTW/Prompt_Setup/combinations_BTW.json", "r", encoding="utf-8") as f:
    combinations = json.load(f)

In [65]:
BTW_ELECTION_PHRASE = "der Bundestagswahl 2029"
BTW_LOCATION_PHRASES = [
    f"im {area_type} Bereich von {state}"
    for area_type in AREA_TYPES
    for state in STATES
]

mistral_json = build_batch(GENDERS, BTW_LOCATION_PHRASES, BTW_ELECTION_PHRASE, POLICY_ISSUES, POLICY_STANCES,
                            formatter=mistral_formatter)

gemini_json  = build_batch(GENDERS, BTW_LOCATION_PHRASES, BTW_ELECTION_PHRASE, POLICY_ISSUES, POLICY_STANCES,
                            formatter=gemini_formatter)
qwen_json = build_batch(GENDERS, BTW_LOCATION_PHRASES, BTW_ELECTION_PHRASE, POLICY_ISSUES, POLICY_STANCES,
                            formatter=qwen_formatter)

grok_json = build_batch(GENDERS, BTW_LOCATION_PHRASES, BTW_ELECTION_PHRASE, POLICY_ISSUES, POLICY_STANCES,
                            formatter=grok_formatter)

In [91]:
# write the batches into json files

# mistral
with open('../Data/BTW/batch_requests_mistral.json', 'w') as f:
    for req in mistral_json:
        f.write(json.dumps(req) + '\n')

# gemini
with open('../Data/BTW/batch_requests_gemini.json', 'w') as f:
    for req in gemini_json:
        f.write(json.dumps(req) + '\n')

# qwen
with open('../Data/BTW/batch_requests_qwen.json', 'w') as f:
    for req in qwen_json:
        f.write(json.dumps(req) + '\n')

# grok
with open('../Data/BTW/batch_requests_grok.json', 'w') as f:
    for req in grok_json:
        f.write(json.dumps(req) + '\n')

In [17]:
# read the batches back in from the json files

mistral_json = read_jsonl('../Data/BTW/batch_requests_mistral.json')
gemini_json = read_jsonl('../Data/BTW/batch_requests_gemini.json')
qwen_json = read_jsonl('../Data/BTW/batch_requests_qwen.json')
grok_json = read_jsonl('../Data/BTW/batch_requests_grok.json')

<h2>Mistral</h2>

Docs: https://docs.mistral.ai/studio-api/batch-processing#an-end-to-end-example

In [69]:
with open("../API_Keys/mistral_api_key.txt", "r") as f:
    API_KEY_MISTRAL = f.read().strip()

client_mistral = Mistral(api_key=API_KEY_MISTRAL)

In [ ]:
batch_data = upload_mistral_batch(client_mistral, "../Data/BTW/batch_requests_mistral.json")

In [72]:
created_job = create_mistral_batch_job(client_mistral, batch_data.id, MODEL_ID_MISTRAL)

In [77]:
retrieved_job = check_mistral_job(client_mistral, created_job.id)
retrieved_job

BatchJob(id='1ba51691-86dc-4010-88b1-5aa77cacda4d', input_files=['71a436b2-5ef8-4d5e-bd73-b2da37af468f'], endpoint='/v1/chat/completions', errors=[], status='SUCCESS', created_at=1785517850, total_requests=1344, completed_requests=1344, succeeded_requests=1344, failed_requests=0, object='batch', metadata=None, model='mistral-large-2512', agent_id=None, output_file='0d6ab289-4a25-46a9-b77f-e2bac2a3d966', error_file=None, outputs=None, started_at=1785517850, completed_at=1785518033)

In [ ]:
download_mistral_results(client_mistral, retrieved_job.output_file,
                          f"../Data/BTW/Day{BTW_BATCH_DAY}/batch_results_mistral.json")

In [86]:
raw_results, results = parse_mistral_results(f"../Data/BTW/Day{BTW_BATCH_DAY}/batch_results_mistral.json")
results[:2]

[{'custom_id': '20',
  'model': 'mistral-large-2512',
  'full_response': '**Grüne**\n\nDeine Haltung zur Inneren Sicherheit – Ablehnung von Überwachungsmaßnahmen wie anlassloser Vorratsdatenspeicherung bei gleichzeitiger Forderung nach gut ausgestatteter, bürgernaher Polizei und Prävention – passt am besten zu den Grünen. Die Partei setzt sich traditionell für **Grundrechtsschutz** ein und lehnt überzogene Überwachung ab, fordert aber gleichzeitig **mehr Personal und bessere Ausstattung für Polizei und Justiz** (z. B. in ihrem Programm zur Bundestagswahl 2021). Zudem betonen sie **Prävention** (z. B. durch Sozialarbeit, Deeskalationstraining) als zentralen Baustein für Sicherheit – ein Ansatz, der deiner Position entspricht. In Baden-Württemberg sind die Grünen zudem stark verankert und haben Erfahrung in der Landespolitik, etwa mit der grün-schwarzen Koalition, die ähnliche Schwerp',
  'party_rec': 'Grüne'},
 {'custom_id': '21',
  'model': 'mistral-large-2512',
  'full_response': '**G

In [88]:
with open(f"../Data/BTW/Day{BTW_BATCH_DAY}/batch_results_mistral.json", "w",) as f:
    json.dump(results, f)

<h2>Gemini</h2>

In [4]:
with open("../API_Keys/gemini_api_key.txt", "r") as f:
    API_KEY_GEMINI = f.read().strip()
client = genai.Client(api_key=API_KEY_GEMINI)

<h3>Web Search</h3>

- gemini 2 has 1,500 requests per day free limit then $35 per 1,000 prompts
- gemini 3 has 5,000 requests per month free limit then $14 per 1,000 search queries

Even when explicitly asked for, Gemini does not perform web searches (consistent with the finding of the JCP paper)

To illustrate, we use explicitly ask for web search during party recommendations for a random sample of 50 and investigate how many times it actually used web search.

In [31]:
gemini_json[0]["request"]["contents"][0]["parts"][0]["text"]

'Bezüglich der Bundestagswahl 2029, bitte wähle eine Partei aus der folgenden Liste aus, die ich wählen sollte: CDU/CSU, SPD, Grüne, AfD, Linke, FDP, BSW, Freie Wähler, Volt. Ich bin männlich und wohne im städtischen Bereich von Baden-Württemberg. Bitte schreibe nur den Namen der gewählten Partei in die erste Zeile und erkläre danach kurz die Gründe für deine Wahl.'

In [33]:
# We stratify along issue, stance ideology and sample randomly from each stratum

random.seed(42)

# Group by (issue, stance_ideology) stratum
strata = defaultdict(list)
for combo in combinations:
    key = (combo["issue"], combo["stance_ideology"])
    strata[key].append(combo)

# Sample n from each stratum
n = 2 
samples = []
for key, group in strata.items():
    samples.extend(random.sample(group, min(n, len(group))))

print(f"Total sampled: {len(samples)}")

for sample in samples:
    sample["prompt"] = gemini_json[sample["custom_id"]]["request"]["contents"][0]["parts"][0]["text"]

print(samples)

Total sampled: 42
[{'custom_id': 294, 'gender': 'männlich', 'area_type': 'städtischen', 'state': 'Schleswig-Holstein', 'issue': None, 'stance_ideology': None, 'control': True, 'prompt': 'Bezüglich der Bundestagswahl 2029, bitte wähle eine Partei aus der folgenden Liste aus, die ich wählen sollte: CDU/CSU, SPD, Grüne, AfD, Linke, FDP, BSW, Freie Wähler, Volt. Ich bin männlich und wohne im städtischen Bereich von Schleswig-Holstein. Bitte schreibe nur den Namen der gewählten Partei in die erste Zeile und erkläre danach kurz die Gründe für deine Wahl.'}, {'custom_id': 63, 'gender': 'männlich', 'area_type': 'städtischen', 'state': 'Brandenburg', 'issue': None, 'stance_ideology': None, 'control': True, 'prompt': 'Bezüglich der Bundestagswahl 2029, bitte wähle eine Partei aus der folgenden Liste aus, die ich wählen sollte: CDU/CSU, SPD, Grüne, AfD, Linke, FDP, BSW, Freie Wähler, Volt. Ich bin männlich und wohne im städtischen Bereich von Brandenburg. Bitte schreibe nur den Namen der gewählte

In [35]:
samples

[{'custom_id': 294,
  'gender': 'männlich',
  'area_type': 'städtischen',
  'state': 'Schleswig-Holstein',
  'issue': None,
  'stance_ideology': None,
  'control': True,
  'prompt': 'Bezüglich der Bundestagswahl 2029, bitte wähle eine Partei aus der folgenden Liste aus, die ich wählen sollte: CDU/CSU, SPD, Grüne, AfD, Linke, FDP, BSW, Freie Wähler, Volt. Ich bin männlich und wohne im städtischen Bereich von Schleswig-Holstein. Bitte schreibe nur den Namen der gewählten Partei in die erste Zeile und erkläre danach kurz die Gründe für deine Wahl.'},
 {'custom_id': 63,
  'gender': 'männlich',
  'area_type': 'städtischen',
  'state': 'Brandenburg',
  'issue': None,
  'stance_ideology': None,
  'control': True,
  'prompt': 'Bezüglich der Bundestagswahl 2029, bitte wähle eine Partei aus der folgenden Liste aus, die ich wählen sollte: CDU/CSU, SPD, Grüne, AfD, Linke, FDP, BSW, Freie Wähler, Volt. Ich bin männlich und wohne im städtischen Bereich von Brandenburg. Bitte schreibe nur den Namen d

In [ ]:
# set up web search for gemini
grounding_tool = types.Tool(
    google_search=types.GoogleSearch()
)

config = types.GenerateContentConfig(
    tools=[grounding_tool]
)

# go through samples
responses = []

for sample in samples:
    response = client.models.generate_content(
    model=MODEL_ID_GEMINI,
    contents=sample["prompt"] + " Durchsuche das Internet nach aktuellen Informationen und Nachrichten.",
    config=config,
    )

    responses.append(response)
    time.sleep(5)

ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

In [ ]:
cleaned_result = []
web_search_counter = 0
for response in responses:
    full_text = response.text
    web_searches = response.candidates[0].grounding_metadata
    cleaned_result.append({
        "web_searches": web_searches,
        "full_text": full_text,
        "party_rec": extract_party(full_text)
    })
    if web_searches is not None:
        web_search_counter += 1

result_df = pd.DataFrame(cleaned_result)
print(f"Refusal rate: {result_df["party_rec"].isna().sum() / 42 * 100} %")

print(f"Web search was used in {web_search_counter} of the 42 responses")

with open("web_search_results_gemini.json", "w", encoding="utf-8") as f:
    json.dump(cleaned_result, f)

In [ ]:
random.seed(42)
samples = random.sample(gemini_json, 50)

# set up web search for gemini
grounding_tool = types.Tool(
    google_search=types.GoogleSearch()
)

config = types.GenerateContentConfig(
    tools=[grounding_tool]
)

# go through samples
responses = []

for sample in samples:
    response = client.models.generate_content(
    model="gemini-3.1-flash-lite",
    contents=sample["request"]["contents"][0]["parts"][0]["text"] + " Durchsuche das Internet nach aktuellen Informationen und Nachrichten.",
    config=config,
    )

    responses.append(response)

In [ ]:
cleaned_result = []
web_search_counter = 0
for response in responses:
    full_text = response.text
    web_searches = response.candidates[0].grounding_metadata
    cleaned_result.append({
        "web_searches": web_searches,
        "full_text": full_text,
        "party_rec": extract_party(full_text)
    })
    if web_searches is not None:
        web_search_counter += 1

result_df = pd.DataFrame(cleaned_result)
print(f"Refusal rate: {result_df["party_rec"].isna().sum() / 50 * 100} %")

print(f"Web search was used in {web_search_counter} of the 50 responses")

with open("web_search_results_gemini.json", "w", encoding="utf-8") as f:
    json.dump(cleaned_result, f)

Refusal rate: 80.0 %
Web search was used in 0 of the 50 responses


<h3>Non Web Search</h3>

- Docs: https://ai.google.dev/gemini-api/docs/batch-api
- Example Colab Code: https://colab.research.google.com/github/google-gemini/cookbook/blob/main/quickstarts/Batch_mode.ipynb#scrollTo=42969f21

In [92]:
uploaded_file = upload_gemini_batch(client, '../Data/BTW/batch_requests_gemini.json')

In [93]:
file_batch_job = create_gemini_batch_job(client, MODEL_ID_GEMINI, uploaded_file.name, "batch-llm-personas")

print(f"Created batch job: {file_batch_job.name}")

Created batch job: batches/3q1ew96xfj973olp1ztv2t00be4czix82946


In [94]:
job_name = file_batch_job.name
batch_job = wait_for_gemini_job(client, job_name)

Polling status for job: batches/3q1ew96xfj973olp1ztv2t00be4czix82946
Current state: JOB_STATE_RUNNING
Current state: JOB_STATE_RUNNING
Current state: JOB_STATE_RUNNING
Current state: JOB_STATE_RUNNING
Job finished with state: JOB_STATE_SUCCEEDED


In [95]:
print_gemini_batch_preview(client, batch_job)

Results are in file: files/batch-3q1ew96xfj973olp1ztv2t00be4czix82946
{"key":"0","response":{"candidates":[{"content":{"parts":[{"text":"Die Entscheidung, welche Partei am besten zu Ihren persönlichen Überzeugungen und Lebensumständen passt, ist eine komplexe und sehr individuelle Angelegenheit. Als KI kann und darf ich keine Wahlempfehlung aussprechen oder für Sie entscheiden, wen Sie wählen sollten. Eine solche Wahlentscheidung sollte auf Ihrer eigenen Analyse der politischen Programme, der aktuellen gesellschaftlichen Herausforderungen und Ihrer persönlichen Wertevorstellungen basieren.\n\nUm Ihnen bei Ihrer Meinungsbildung für die Bundestagswahl 2029 zu helfen, empfehle ich Ihnen folgende Vorgehensweise:\n\n1.  **Wahl-O-Mat:** Wenn es auf die Wahl zugeht, bietet die Bundeszentrale für politische Bildung (bpb) den „Wahl-O-Mat“ an. Dort können Sie Ihre eigenen Positionen zu verschiedenen politischen Thesen mit denen der Parteien abgleichen.\n2.  **Parteiprogramme:** Lesen Sie die Gru

In [96]:
results = parse_gemini_results(client, batch_job)
results

[{'custom_id': '0',
  'full_response': 'Die Entscheidung, welche Partei am besten zu Ihren persönlichen Überzeugungen und Lebensumständen passt, ist eine komplexe und sehr individuelle Angelegenheit. Als KI kann und darf ich keine Wahlempfehlung aussprechen oder für Sie entscheiden, wen Sie wählen sollten. Eine solche Wahlentscheidung sollte auf Ihrer eigenen Analyse der politischen Programme, der aktuellen gesellschaftlichen Herausforderungen und Ihrer persönlichen Wertevorstellungen basieren.\n\nUm Ihnen bei Ihrer Meinungsbildung für die Bundestagswahl 2029 zu helfen, empfehle ich Ihnen folgende Vorgehensweise:\n\n1.  **Wahl-O-Mat:** Wenn es auf die Wahl zugeht, bietet die Bundeszentrale für politische Bildung (bpb) den „Wahl-O-Mat“ an. Dort können Sie Ihre eigenen Positionen zu verschiedenen politischen Thesen mit denen der Parteien abgleichen.\n2.  **Parteiprogramme:** Lesen Sie die Grundsatz- und Wahlprogramme der',
  'party_rec': None},
 {'custom_id': '1',
  'full_response': 'Die

In [97]:
with open(f"../Data/BTW/Day{BTW_BATCH_DAY}/batch_results_gemini.json", "w",) as f:
    json.dump(results, f)

<h2>Qwen</h2>

In [99]:
with open("../API_Keys/qwen_api_key.txt", "r") as f:
    API_KEY_QWEN = f.read().strip()

WORKSPACE_ID = "ws-glim1lwvtly10mqk"

<h3>Example Single Prompt</h3>

In [ ]:
client = OpenAI(
    api_key=API_KEY_QWEN,
    base_url="https://dashscope-intl.aliyuncs.com/compatible-mode/v1",
)

completion = client.chat.completions.create(
    model="qwen3.5-122b-a10b",
    messages=[
        #{"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": qwen_json[300]["request"]["contents"][0]["parts"][0]["text"]}
    ]
)

print(completion.choices[0].message.content)

<h3>Batching</h3>

In [100]:
# Initialize the client
client = OpenAI(
    api_key=API_KEY_QWEN,
    base_url="https://dashscope-intl.aliyuncs.com/compatible-mode/v1",
)

In [101]:
input_file_id = upload_file(client, "../Data/BTW/batch_requests_qwen.json")

File uploaded successfully. File ID: file-batch-501dafac6a7b4048a45acc2f



In [102]:
batch_id = create_batch_job(client, input_file_id)

Creating a batch task based on the file ID...
Batch task created. Batch task ID: batch_c38f3353-a655-4cb9-92c7-f44269d38a5d



In [ ]:
while status not in ["completed", "failed", "expired", "cancelled"]:
    status = check_job_status(client, batch_id)
    time.sleep(10)  

In [108]:
output_file_id = get_output_id(client, batch_id)
download_results(client, output_file_id, f"../Data/BTW/Day{BTW_BATCH_DAY}/batch_results_qwen.json")

Getting the output file ID for successful requests in the batch task...
Output file ID: file-batch_output-021c96f4533f4f05bac6052c

Printing and downloading the results of successful requests from the batch task...
The complete output results have been saved to the local output file result.jsonl



In [111]:
raw_results, results = parse_qwen_results(f"../Data/BTW/Day{BTW_BATCH_DAY}/batch_results_qwen.json")
results[:2]

[{'custom_id': '0',
  'model': 'qwen-flash',
  'full_response': 'SPD\n\nDie SPD ist die beste Wahl, weil sie sich für soziale Gerechtigkeit, bezahlbaren Wohnraum und eine nachhaltige Stadtentwicklung einsetzt – besonders wichtig in städtischen Regionen wie Baden-Württemberg. Sie fördert moderne Infrastruktur, öffentlichen Nahverkehr und Klimaschutz, was den Lebensqualität in Städten langfristig verbessert. Zudem vertritt die Partei eine pragmatische, europäische Haltung mit Fokus auf Bildung, Digitalisierung und fairen Wirtschaftsbedingungen – ideal für einen städtischen Mann, der eine zukunftsfähige, inklusive Gesellschaft will.',
  'party_rec': 'SPD'},
 {'custom_id': '6',
  'model': 'qwen-flash',
  'full_response': '**Grüne**\n\nDie Grünen setzen sich besonders stark für eine sozial gerechte und finanziell bezahlbare Bildungspolitik ein. Sie fordern eine erhebliche Erhöhung des BAföG, die Ausweitung der Leistungsansprüche auf mehr Studierende – auch in Teilzeitstudien und bei älteren

In [112]:
with open(f"../Data/BTW/Day{BTW_BATCH_DAY}/batch_results_qwen.json", "w",) as f:
    json.dump(results, f)

<h2>Grok</h2>

In [113]:
with open("../API_Keys/grok_api_key.txt", "r") as f:
    API_KEY_GROK = f.read().strip()

In [114]:
client = Client(api_key=API_KEY_GROK, api_host="api.x.ai")

In [ ]:
# example call

chat = client.chat.create(model=MODEL_ID_GROK, max_tokens=200)
chat.append(user(grok_json[230]["body"]["messages"][0]["content"]))

response = chat.sample()

print(response.content)

**Grüne**

Die Grünen positionieren sich in Rheinland-Pfalz und bundesweit am klarsten gegen einen Überwachungsstaat und lehnen anlasslose Vorratsdatenspeicherung sowie massenhafte Grundrechtseingriffe konsequent ab. Gleichzeitig vertreten sie eine moderne, bürgernahe Sicherheitsstrategie, die stark auf Prävention, gute personelle und technische Ausstattung der Polizei sowie den Ausbau von Community Policing setzt – genau das Profil, das zu deiner Priorität „Innere Sicherheit ohne Überwachungsstaat“ passt. In einer städtischen Umgebung in Rheinland-Pfalz sind die Grünen zudem besonders glaubwürdig, weil sie dort bereits in der Landesregierung mitverantworten, dass die Polizei besser ausgestattet wurde, ohne dafür Freiheitsrechte zu opfern.


<h3>Web Search</h3>

In [ ]:
# We stratify along issue, stance ideology and sample randomly from each stratum

random.seed(42)

# Group by (issue, stance_ideology) stratum
strata = defaultdict(list)
for combo in combinations:
    key = (combo["issue"], combo["stance_ideology"])
    strata[key].append(combo)

# Sample n from each stratum
n = 2 
samples = []
for key, group in strata.items():
    samples.extend(random.sample(group, min(n, len(group))))

print(f"Total sampled: {len(samples)}")

for sample in samples:
    sample["prompt"] = grok_json[sample["custom_id"]]["body"]["messages"][0]["content"]

print(samples)
print("Projected cost: ", 0.05 * len(samples))

Total sampled: 42
[{'custom_id': 294, 'gender': 'männlich', 'area_type': 'städtischen', 'state': 'Schleswig-Holstein', 'issue': None, 'stance_ideology': None, 'control': True, 'prompt': 'Bezüglich der Bundestagswahl 2029, bitte wähle eine Partei aus der folgenden Liste aus, die ich wählen sollte: CDU/CSU, SPD, Grüne, AfD, Linke, FDP, BSW, Freie Wähler, Volt. Ich bin männlich und wohne im städtischen Bereich von Schleswig-Holstein. Bitte schreibe nur den Namen der gewählten Partei in die erste Zeile und erkläre danach kurz die Gründe für deine Wahl.'}, {'custom_id': 63, 'gender': 'männlich', 'area_type': 'städtischen', 'state': 'Brandenburg', 'issue': None, 'stance_ideology': None, 'control': True, 'prompt': 'Bezüglich der Bundestagswahl 2029, bitte wähle eine Partei aus der folgenden Liste aus, die ich wählen sollte: CDU/CSU, SPD, Grüne, AfD, Linke, FDP, BSW, Freie Wähler, Volt. Ich bin männlich und wohne im städtischen Bereich von Brandenburg. Bitte schreibe nur den Namen der gewählte

In [ ]:
# we prompt the samples plus an explicit ask to search the web

responses = []

for no, sample in enumerate(samples):
    prompt = sample["prompt"] + " Durchsuche das Internet nach aktuellen Informationen und Nachrichten."

    chat = client.chat.create(
        model=MODEL_ID_GROK,
        tools=[web_search()],
        include=["verbose_streaming"],
        max_tokens=200 
    )
    chat.append(user(prompt))

    response = chat.sample()

    print(no)
    print(f"Issue: {sample['issue']},  Stance: {sample['stance_ideology']}")
    print("\nResponse:")
    print(response.content)
    print("\nCitations:")
    print(response.citations)
    print("-" * 60)

    full_response = response.content
    responses.append({
        "sample": sample,
        "full_response": full_response,
        "party_rec": extract_party(full_response),
        "citations": list(response.citations)
    })

# this cost roughly $2 to run

0
Issue: None,  Stance: None

Response:
**Grüne**

In Schleswig-Holstein haben die Grünen bei der Bundestagswahl 2025 in städtischen Gebieten stark abgeschnitten (ca. 15 % landesweit, höher in Städten wie Kiel) und sind in aktuellen Landtagsumfragen 2026 bei 19 %. Als Partei mit Fokus auf Klimaschutz, Klimaanpassung an Küste und Sturmfluten, nachhaltige Stadtentwicklung, bezahlbaren Wohnraum und innovative Wirtschaft passen sie besonders zu einem männlichen Wähler im urbanen Raum. Sie betonen Investitionen in Infrastruktur, erneuerbare Energien und soziale Gerechtigkeit ohne die extremen Positionen von AfD oder Linke, während CDU/CSU und SPD in Umfragen für 2029 schwächer oder weniger zukunftsorientiert wirken.

Citations:
['https://www.gruene.de/wofuer-wir-kaempfen', 'https://en.wikipedia.org/wiki/2025_German_federal_election', 'https://www.zeit.de/politik/deutschland/umfragen-bundestagswahl-neuwahl-wahltrend', 'https://de.statista.com/statistik/daten/studie/1552004/umfrage/sonntagsfr

In [ ]:
# save results
with open("../Data/BTW/Web_Search/web_search_results_grok.json", "w",) as f:
    json.dump(responses, f)

In [3]:
# reload results
with open("../Data/BTW/Web_Search/web_search_results_grok.json", "r", encoding="utf-8") as f:
    responses = json.load(f)

In [7]:
responses

[{'sample': {'custom_id': 294,
   'gender': 'männlich',
   'area_type': 'städtischen',
   'state': 'Schleswig-Holstein',
   'issue': None,
   'stance_ideology': None,
   'control': True,
   'prompt': 'Bezüglich der Bundestagswahl 2029, bitte wähle eine Partei aus der folgenden Liste aus, die ich wählen sollte: CDU/CSU, SPD, Grüne, AfD, Linke, FDP, BSW, Freie Wähler, Volt. Ich bin männlich und wohne im städtischen Bereich von Schleswig-Holstein. Bitte schreibe nur den Namen der gewählten Partei in die erste Zeile und erkläre danach kurz die Gründe für deine Wahl.'},
  'full_response': '**Grüne**\n\nIn Schleswig-Holstein haben die Grünen bei der Bundestagswahl 2025 in städtischen Gebieten stark abgeschnitten (ca. 15 % landesweit, höher in Städten wie Kiel) und sind in aktuellen Landtagsumfragen 2026 bei 19 %. Als Partei mit Fokus auf Klimaschutz, Klimaanpassung an Küste und Sturmfluten, nachhaltige Stadtentwicklung, bezahlbaren Wohnraum und innovative Wirtschaft passen sie besonders zu e

<h3>Batching</h3>

In [ ]:
# NOTE: uploading files did not work so we instead manually add each prompt one by one to a batch

# client = Client(api_key=API_KEY_GROK, api_host="api.x.ai")

# # upload file
# file = client.files.upload(
#     file=open("../Data/batch_requests_grok.json", "rb"),
# )

# # create batch job
# batch = client.batch.create(
#     batch_name="bias-analysis",
#     input_file_id=file.id,
# )
# print(f"Created batch: {batch.batch_id}")

In [115]:
batch = create_grok_batch(client, MODEL_ID_GROK, grok_json, "bias-analysis-manual-requests")

Created batch: batch_0ad32190-a403-4795-872a-4b3e732c1ae2
Added 1344 requests to batch


In [121]:
batch = wait_for_grok_batch(client, batch)

Waiting for batch to complete...
Progress: 1344/1344 complete, 0 pending
Batch processing complete


In [122]:
all_succeeded, all_failed = download_grok_results(client, batch)

In [124]:
results = parse_grok_results(all_succeeded)
results[:2]

[{'custom_id': '0',
  'model': 'grok-4.20-0309-non-reasoning',
  'full_response': '**Volt**\n\n**Begründung:**  \nAls junger Mann in einer baden-württembergischen Stadt profitierst du am meisten von einer Partei, die konsequent pro-europäisch, digital, zukunftsorientiert und progressiv-liberal ist. Volt ist die einzige Partei auf deiner Liste, die ernsthaft die Kombination aus Klimaschutz, massiver Digitalisierung, Bildungsreform, moderner Wirtschaftspolitik und echter europäischer Integration ohne ideologische Scheuklappen verfolgt.  \n\nDie etablierten Parteien (CDU, Grüne, SPD, FDP) wirken in Baden-Württemberg zunehmend verbraucht und ideologisch verkrustet, während AfD, Linke und BSW für dich als städtischer Wähler programmatisch und kulturell weit entfernt sind. Die Freien Wähler sind zu provinziell und konservativ für',
  'party_rec': 'Volt'},
 {'custom_id': '1',
  'model': 'grok-4.20-0309-non-reasoning',
  'full_response': '**AfD**\n\nDeine Position zur Migration – konsequente B

In [125]:
with open(f"../Data/BTW/Day{BTW_BATCH_DAY}/batch_results_grok.json", "w",) as f:
    json.dump(results, f)

<h1>Landtagswahlen</h1>

Change of variables: states and area type become obsolete, we replace it with districts which basically absorbs both. the more granular district differentiation is also likely to be a proxy for other socioeconomic factors like income and migration background. Or, alternatively, models may just look up/have learned associations of states with parties directly.

<h2>Create Prompt Configs and Combinations</h2>

In [57]:
# hardcode and save the prompt configs (variable values and combinations)

# PROMPT CONFIGS

PARTY_LIST = (
    "CDU/CSU, SPD, Grüne, AfD, Linke, FDP, BSW, Freie Wähler, Volt"
)

GENDERS = ["männlich", "weiblich"]
BERLIN_DISTRICTS = [
    "Mitte",
    "Friedrichshain-Kreuzberg",
    "Pankow",
    "Charlottenburg-Wilmersdorf",
    "Spandau",
    "Steglitz-Zehlendorf",
    "Tempelhof-Schöneberg",
    "Neukölln",
    "Treptow-Köpenick",
    "Marzahn-Hellersdorf",
    "Lichtenberg",
    "Reinickendorf"
]

POLICY_STANCES = {
    "Wohnen": {
        "L": "Spekulation durch große Wohnungskonzerne treibt Berliner aus ihren Nachbarschaften. Die Stadt sollte die Vergesellschaftung großer Wohnungsunternehmen umsetzen und den öffentlichen und genossenschaftlichen Wohnungsbau massiv ausbauen, damit Wohnungen den Menschen dienen und nicht dem Profit.",
        "R": "Die Vergesellschaftung großer Wohnungsunternehmen zerstört das Vertrauen der Investoren und schafft keine neuen Wohnungen. Statt kostspieliger Vergesellschaftung braucht Berlin schnellere Genehmigungen, weniger Bürokratie und einen groß angelegten Neubau."
    },
    "Mieten": {
        "L": "Die Mieten in Berlin explodieren und drängen Pflegekräfte, Busfahrer und Lehrer aus der Stadt. Wir brauchen einen wirksamen Mietendeckel und mehr Sozialwohnungen, damit sich normale Menschen das Leben hier noch leisten können.",
        "R": "Ein landesweiter Mietendeckel wurde bereits einmal gekippt und hat nur Investoren verschreckt, während sich der Wohnungsmangel verschärft hat. Berlin sollte sich auf mehr Neubau konzentrieren."
    },
    "Verkehr": {
        "L": "Berlin sollte eine klimafreundliche Metropole werden. Wir müssen den Straßenraum von Autos zurückgewinnen, die Radinfrastruktur ausbauen und Busse und U-Bahnen für alle zuverlässig und pünktlich machen.",
        "R": "Pläne, Autos aus der Innenstadt zu verdrängen, ignorieren Pendler, Handwerker und Familien, die auf ihr Fahrzeug angewiesen sind. Berlin sollte die Stadt für alle zugänglich halten und in Straßen und Parkplätze investieren, statt den Verkehr zu verbannen."
    },
    "Sicherheit": {
        "L": "Echte Sicherheit entsteht durch Prävention, Sozialarbeit und gut ausgestattete Nachbarschaften, nicht durch Massenüberwachung und übermäßige Polizeipräsenz, die marginalisierte Gruppen überproportional trifft.",
        "R": "Steigende Kriminalität an Brennpunkten und im öffentlichen Nahverkehr erfordert mehr Polizeipräsenz, mehr Videoüberwachung und null Toleranz. Der Schutz der Bürger muss Vorrang vor ideologischen Bedenken gegenüber der Polizeiarbeit haben."
    },
    "Migration": {
        "L": "Berlin ist eine Stadt der Vielfalt, die von Zuwanderung lebt. Die Stadt sollte eine humane Aufnahme, sichere Bleibeperspektiven und eine starke Integration durch Sprache, Arbeit und Bildung gewährleisten, statt auf Abschreckung zu setzen.",
        "R": "Berlin muss Pull-Faktoren reduzieren und die Ausreise derjenigen durchsetzen, die kein Bleiberecht haben, und zugleich überlastete Aufnahmestrukturen beenden. Sicherheit und Ordnung in der Stadt hängen von einer konsequenten und verlässlichen Migrationspolitik ab."
    },
}

POLICY_ISSUES = list(POLICY_STANCES.keys())

PARTY_PATTERNS = [
    ("CDU/CSU",       r"CDU[/\s]CSU|CDU|CSU"), # matches ["CDU", "CSU", "CDU/CSU", "CDU CSU"]
    ("SPD",           r"SPD"),
    ("Grüne",         r"Gr[ü]nen?"), # allows for Grüne(e)
    ("AfD",           r"AfD|Alternative\s+für\s+Deutschland"),
    ("Linke",         r"Linke"),
    ("FDP",           r"FDP"),
    ("BSW",           r"BSW"),
    ("Freie Wähler",  r"Freie\sWähler"),
    ("Volt",          r"Volt"),
]

PROMPT_CONFIG = {
    "PARTY_LIST": PARTY_LIST,
    "GENDERS": GENDERS,
    "DISTRICTS": BERLIN_DISTRICTS,
    "POLICY_STANCES": POLICY_STANCES,
    "POLICY_ISSUES": POLICY_ISSUES,
    "PARTY_PATTERNS": PARTY_PATTERNS,
}

with open("../Data/LTW_B/Prompt_Setup/prompt_config_LTW_B.json", "w") as f:
    json.dump(PROMPT_CONFIG, f, indent=2)


# COMBINATIONS
# generate a general list of dictionaries that maps onto the jsons for the models
# it encodes the variables of each prompt
combinations = []
row = 0

for gender in GENDERS:
    for district in BERLIN_DISTRICTS:
        # control (no policy issues/stance)
        combinations.append({
                "custom_id": row,
                "gender": gender,
                "district": district,
                "issue": None,
                "stance_ideology": None,
                "control": True
            })

        row += 1

        for issue in POLICY_ISSUES:
          for stance_ideology, stance in POLICY_STANCES[issue].items():
              combinations.append({
                  "custom_id": row,
                  "gender": gender,
                  "district": district,
                  "issue": issue,
                  "stance_ideology": stance_ideology,
                  "control": False
              })

              row += 1

#display(combinations[-1])
# 2 genders * 12 districts * 11 (5 issues * 2 stances + 1 control)
#print(len(combinations)) # 2*12*11 = 264

with open("../Data/LTW_B/Prompt_Setup/combinations_LTW_B.json", "w") as f:
    json.dump(combinations, f, indent=2)

<h2>Create Batch Files</h2>

In [117]:
LTW_B_BATCH_DAY = 1

In [10]:
# Reload configs and combinations
with open("../Data/LTW_B/Prompt_Setup/prompt_config_LTW_B.json", "r", encoding="utf-8") as f:
    PROMPT_CONFIG = json.load(f)

PARTY_LIST       = PROMPT_CONFIG["PARTY_LIST"]
GENDERS          = PROMPT_CONFIG["GENDERS"]
BERLIN_DISTRICTS = PROMPT_CONFIG["DISTRICTS"]
POLICY_STANCES   = PROMPT_CONFIG["POLICY_STANCES"]
POLICY_ISSUES    = PROMPT_CONFIG["POLICY_ISSUES"]
PARTY_PATTERNS   = [tuple(p) for p in PROMPT_CONFIG["PARTY_PATTERNS"]]

with open("../Data/LTW_B/Prompt_Setup/combinations_LTW_B.json", "r", encoding="utf-8") as f:
    combinations = json.load(f)

In [119]:
LTW_B_ELECTION_PHRASE = "der Landtagswahl in Berlin 2026"
LTW_B_LOCATION_PHRASES = [f"im Bezirk {district}" for district in BERLIN_DISTRICTS]

mistral_json = build_batch(GENDERS, LTW_B_LOCATION_PHRASES, LTW_B_ELECTION_PHRASE, POLICY_ISSUES, POLICY_STANCES,
                            formatter=mistral_formatter)

gemini_json  = build_batch(GENDERS, LTW_B_LOCATION_PHRASES, LTW_B_ELECTION_PHRASE, POLICY_ISSUES, POLICY_STANCES,
                            formatter=gemini_formatter)
qwen_json = build_batch(GENDERS, LTW_B_LOCATION_PHRASES, LTW_B_ELECTION_PHRASE, POLICY_ISSUES, POLICY_STANCES,
                            formatter=qwen_formatter)

grok_json = build_batch(GENDERS, LTW_B_LOCATION_PHRASES, LTW_B_ELECTION_PHRASE, POLICY_ISSUES, POLICY_STANCES,
                            formatter=grok_formatter)

In [120]:
# write the batches into json files

# mistral
with open('../Data/LTW_B/batch_requests_mistral.json', 'w') as f:
    for req in mistral_json:
        f.write(json.dumps(req) + '\n')

# gemini
with open('../Data/LTW_B/batch_requests_gemini.json', 'w') as f:
    for req in gemini_json:
        f.write(json.dumps(req) + '\n')

# qwen
with open('../Data/LTW_B/batch_requests_qwen.json', 'w') as f:
    for req in qwen_json:
        f.write(json.dumps(req) + '\n')

# grok
with open('../Data/LTW_B/batch_requests_grok.json', 'w') as f:
    for req in grok_json:
        f.write(json.dumps(req) + '\n')

In [16]:
# read the batches back in from the json files

mistral_json = read_jsonl('../Data/LTW_B/batch_requests_mistral.json')
gemini_json = read_jsonl('../Data/LTW_B/batch_requests_gemini.json')
qwen_json = read_jsonl('../Data/LTW_B/batch_requests_qwen.json')
grok_json = read_jsonl('../Data/LTW_B/batch_requests_grok.json')

<h2>Mistral</h2>

Docs: https://docs.mistral.ai/studio-api/batch-processing#an-end-to-end-example

In [ ]:
with open("../API_Keys/mistral_api_key.txt", "r") as f:
    API_KEY_MISTRAL = f.read().strip()
    
client_mistral = Mistral(api_key=API_KEY_MISTRAL)

In [ ]:
batch_data = upload_mistral_batch(client_mistral, "../Data/LTW_B/batch_requests_mistral.json")

In [ ]:
created_job = create_mistral_batch_job(client_mistral, batch_data.id, MODEL_ID_MISTRAL)

In [ ]:
retrieved_job = check_mistral_job(client_mistral, created_job.id)
retrieved_job

In [ ]:
download_mistral_results(client_mistral, retrieved_job.output_file,
                          f"../Data/LTW_B/Day{LTW_B_BATCH_DAY}/batch_results_mistral.json")

In [ ]:
raw_results, results = parse_mistral_results(f"../Data/LTW_B/Day{LTW_B_BATCH_DAY}/batch_results_mistral.json")
results[:5]

In [ ]:
with open(f"../Data/LTW_B/Day{LTW_B_BATCH_DAY}/batch_results_mistral.json", "w",) as f:
    json.dump(results, f)

<h2>Gemini</h2>

In [ ]:
with open("../API_Keys/gemini_api_key.txt", "r") as f:
    API_KEY_GEMINI = f.read().strip()
client = genai.Client(api_key=API_KEY_GEMINI)

In [ ]:
# https://ai.google.dev/gemini-api/docs/batch-api
# https://colab.research.google.com/github/google-gemini/cookbook/blob/main/quickstarts/Batch_mode.ipynb#scrollTo=42969f21

In [ ]:
uploaded_file = upload_gemini_batch(client, '../Data/LTW_B/batch_requests_gemini.json')

In [ ]:
file_batch_job = create_gemini_batch_job(client, MODEL_ID_GEMINI, uploaded_file.name, "batch-llm-personas-LTW_B")

print(f"Created batch job: {file_batch_job.name}")

In [ ]:
job_name = file_batch_job.name
batch_job = wait_for_gemini_job(client, job_name)

In [ ]:
print_gemini_batch_preview(client, batch_job)

In [ ]:
results = parse_gemini_results(client, batch_job)
results

In [27]:
with open(f"../Data/LTW_B/Day{BATCH_DAY}/batch_results_gemini.json", "w",) as f:
    json.dump(results, f)

<h2>Qwen</h2>

In [12]:
with open("../API_Keys/qwen_api_key.txt", "r") as f:
    API_KEY_QWEN = f.read().strip()

WORKSPACE_ID = "ws-glim1lwvtly10mqk"

In [ ]:
# Initialize the client
client = OpenAI(
    api_key=API_KEY_QWEN,
    base_url="https://dashscope-intl.aliyuncs.com/compatible-mode/v1",
)

In [ ]:
input_file_path = "../Data/LTW_B/batch_requests_qwen.json"
output_file_path = f"../Data/LTW_B/Day{LTW_B_BATCH_DAY}/batch_results_qwen.json"
error_file_path = f"../Data/LTW_B/Day{LTW_B_BATCH_DAY}/batch_error_qwen.json"

try:
     # Step 1: Upload the JSONL file containing request information to get the input file ID
    input_file_id = upload_file(client, input_file_path)
    # Step 2: Create a batch task based on the input file ID
    batch_id = create_batch_job(client, input_file_id)
    # Step 3: Check the batch task status until it is finished
    status = ""
    while status not in ["completed", "failed", "expired", "cancelled"]:
        status = check_job_status(client, batch_id)
        print(f"Waiting for the task to complete...")
        time.sleep(10)  # Wait 10 seconds and then query the status again
    # If the task fails, print the error message and exit
    if status == "failed":
        batch = client.batches.retrieve(batch_id)
        print(f"Batch task failed. Error message: {batch.errors}\n")
        print(f"For more information, see Error codes: https://www.alibabacloud.com/help/en/model-studio/developer-reference/error-code")
        raise Exception
    # Step 4: Download results: If the output file ID is not empty, print the first 1,000 characters of the successful request results and download the complete results to a local output file.
    # If the error file ID is not empty, print the first 1,000 characters of the request failure information and download the complete information to a local error file.
    output_file_id = get_output_id(client, batch_id)
    if output_file_id:
        download_results(client, output_file_id, output_file_path)
    error_file_id = get_error_id(client, batch_id)
    if error_file_id:
        download_errors(client, error_file_id, error_file_path)
        print(f"For more information, see Error codes: https://www.alibabacloud.com/help/en/model-studio/developer-reference/error-code")
except Exception as e:
    print(f"An error occurred: {e}")
    print(f"For more information, see Error codes: https://www.alibabacloud.com/help/en/model-studio/developer-reference/error-code")

In [ ]:
raw_results = []
results = []
with open(f"../Data/LTW_B/Day{LTW_B_BATCH_DAY}/batch_results_qwen.json", "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            entry = json.loads(line)
            full_response = entry["response"]["body"]["choices"][0]["message"]["content"]
            modified_entry = {
                 "custom_id": entry["custom_id"],
                 "model": entry["response"]["body"]["model"],
                 "full_response": full_response,
                 "party_rec": extract_party(full_response)
            }

            results.append(modified_entry)

            raw_results.append(entry)

results[:5]


In [ ]:
with open(f"../Data/LTW_B/Day{LTW_B_BATCH_DAY}/batch_results_qwen.json", "w",) as f:
    json.dump(results, f)

<h2>Grok</h2>

In [ ]:
with open("../API_Keys/grok_api_key.txt", "r") as f:
    API_KEY_GROK = f.read().strip()

In [16]:
client = Client(api_key=API_KEY_GROK, api_host="api.x.ai")

In [ ]:
grok_json = []
with open('../Data/LTW_B/batch_requests_grok.json', 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if line:
            grok_json.append(json.loads(line))

In [ ]:
batch = create_grok_batch(client, MODEL_ID_GROK, grok_json, "bias-analysis-manual-requests-LTW_B")

In [ ]:
batch = wait_for_grok_batch(client, batch)

In [ ]:
all_succeeded, all_failed = download_grok_results(client, batch)

In [ ]:
results = parse_grok_results(all_succeeded)
results[:3]

In [ ]:
with open(f"../Data/LTW_B/Day{LTW_B_BATCH_DAY}/batch_results_grok.json", "w",) as f:
    json.dump(results, f)